In [27]:
# CS340 Project 2
# Becca Hammond 10/19/2025

# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# Import CRUD Python module
from CRUD_Python_Module_5 import AnimalShelter

###########################
# Data Manipulation / Model
###########################
username = "aacuser"
password = "MmmFood"

# Connect to database via CRUD Module
shelter = AnimalShelter(username, password, host='localhost', port=27017, db_name='aac', collection_name='animals')

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(shelter.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'],inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

# Add in Grazioso Salvare’s logo
image_filename = 'Grazioso Salvare Logo.png' 
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

# Place the HTML image tag and include a unique identifier 
app.layout = html.Div([
#    html.Div(id='hidden-div', style={'display':'none'}),
    html.Center(html.B(html.H1('CS-340 Dashboard - Grazioso Salvare - Becca Hammond - 10/19/2025'))),
        html.Center(
        html.A(
            html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()), 
                     style={'height': '150px'}),  # Changed to 150px (half of ~300px)
            href='http://www.snhu.edu',
            target='_blank'
        )
    ),
    html.Hr(),
    # Interactive Filter Options
    html.Div([
        html.Label('Filter by Rescue Type:', style={'fontWeight': 'bold', 'fontSize': '18px', 'marginBottom': '10px'}),
        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': ' Water Rescue', 'value': 'Water'},
                {'label': ' Mountain or Wilderness Rescue', 'value': 'Mountain'},
                {'label': ' Disaster Rescue or Individual Tracking', 'value': 'Disaster'},
                {'label': ' Reset', 'value': 'Reset'}
            ],
            value='Reset',
            labelStyle={'display': 'block', 'marginTop': '10px'},
            style={'fontSize': '16px'}
        )
    ], style={'margin': '20px', 'padding': '20px', 'border': '2px solid #cccccc', 'borderRadius': '5px', 'backgroundColor': '#fafafa'}),
    
    html.Hr(),
    dash_table.DataTable(
        id='datatable-id',
        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
        data=df.to_dict('records'),
        # User-friendly features from Module 6
        row_selectable="single",
        selected_rows=[0],
        page_size=10,
        sort_action="native",
        filter_action="native",
        page_action="native",
        style_table={'overflowX': 'auto'},
        style_cell={
            'textAlign': 'left',
            'padding': '5px',
            'minWidth': '100px'
        },
        style_header={
            'backgroundColor': 'rgb(230, 230, 230)',
            'fontWeight': 'bold'
        }
    ),
    html.Br(),
    html.Hr(),
#This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(className='row',
         style={'display' : 'flex'},
             children=[
        html.Div(
            id='graph-id',
            className='col s12 m6',

            ),
        html.Div(
            id='map-id',
            className='col s12 m6',
            style={'height': '500px'} 
            )
        ])
])

#############################################
# Interaction Between Components / Controller
#############################################    

@app.callback(Output('datatable-id','data'),
              [Input('filter-type', 'value')]
 )
def update_dashboard(filter_type):
    ## code to filter interactive data table with MongoDB queries
    # Filter data based on rescue type selection.
    # Uses MongoDB queries via CRUD module to retrieve filtered data.
    
    if filter_type == 'Water':
        # Water Rescue criteria
        query = {
            'breed': {'$in': ['Labrador Retriever Mix', 'Chesapeake Bay Retriever', 'Newfoundland']},
            'sex_upon_outcome': 'Intact Female',
            'age_upon_outcome_in_weeks': {'$gte': 26, '$lte': 156}
        }
    elif filter_type == 'Mountain':
        # Mountain or Wilderness Rescue criteria
        query = {
            'breed': {'$in': ['German Shepherd', 'Alaskan Malamute', 'Old English Sheepdog', 
                             'Siberian Husky', 'Rottweiler']},
            'sex_upon_outcome': 'Intact Male',
            'age_upon_outcome_in_weeks': {'$gte': 26, '$lte': 156}
        }
    elif filter_type == 'Disaster':
        # Disaster or Individual Tracking criteria
        query = {
            'breed': {'$in': ['Doberman Pinscher', 'German Shepherd', 'Golden Retriever',
                             'Bloodhound', 'Rottweiler']},
            'sex_upon_outcome': 'Intact Male',
            'age_upon_outcome_in_weeks': {'$gte': 20, '$lte': 300}
        }
    else:  # Reset
        query = {}
    
    # Query database using CRUD module
    filtered_data = shelter.read(query)
    
    # Convert to DataFrame and remove _id
    dff = pd.DataFrame.from_records(filtered_data)
    if not dff.empty and '_id' in dff.columns:
        dff.drop(columns=['_id'], inplace=True)
    
    return dff.to_dict('records')

# Display the breeds of animal based on quantity represented in
# the data table
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")]
)
def update_graphs(viewData):
    # Create a pie chart showing breed distribution of filtered data
    
    if not viewData:
        return [html.Div("No data to display")]
    
    dff = pd.DataFrame.from_dict(viewData)
    
    # Get top 10 breeds
    breed_counts = dff['breed'].value_counts().head(10)
    
    return [
        dcc.Graph(            
            figure=px.pie(
                values=breed_counts.values,
                names=breed_counts.index,
                title='Top 10 Preferred Breeds',
                hole=0.3  # Makes it a donut chart for better readability
            )
        )    
    ]
    
#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_rows')]
)
def update_styles(selected_rows):
    if not selected_rows:
        return []
    
    return [{
        'if': {'row_index': i},  
        'background_color': '#D2F3FF'
    } for i in selected_rows] 

# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):  
    if not viewData:
        return [html.Div("No data available")]
    
    dff = pd.DataFrame.from_dict(viewData)
    # Because we only allow single row selection, the list can be converted to a row index here
    if index is None or len(index) == 0:
        row = 0
    else: 
        row = index[0]
        
    # Validate row index
    if row >= len(dff):
        row = 0
    
    # Get location data using column names
    try:
        lat = dff.loc[row, 'location_lat']
        lon = dff.loc[row, 'location_long']
        breed = dff.loc[row, 'breed']
        name = dff.loc[row, 'name']
    except (IndexError, KeyError) as e:
        print(f"Error accessing data: {e}")
        return [html.Div("Map data not available")]
        
    # Austin TX is at [30.75,-97.48]
    return [
        dl.Map(style={'width': '100px', 'height': '500px'}, center=[30.75,-97.48], zoom=10, children=[
            dl.TileLayer(id="base-layer-id"),
            # Marker with tool tip and popup
            # Column 13 and 14 define the grid-coordinates for the map
            # Column 4 defines the breed for the animal
            # Column 9 defines the name of the animal
            dl.Marker(position=[lat, lon], children=[
                dl.Tooltip(breed),
                dl.Popup([
                    html.H1("Animal Name"),
                    html.P(name)
                ])
            ])
        ])
    ]


# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server() 

Dash app running on https://weatherdilemma-prettysalsa-3000.codio.io/proxy/8050/
